| Metric                      | What it measures                                 | How to compute                               | What you need                |
| --------------------------- | ------------------------------------------------ | -------------------------------------------- | ---------------------------- |
| **Numeric coverage MAE**    | Accuracy of cloud cover values (`hcc, mcc, lcc`) | Compare values extracted from text vs. CSV   | CSV + regex                  |
| **Cloud classification F1** | Correct presence/absence of high/mid/low clouds  | CSV >0.1 → present; search keywords in text  | CSV + keyword list           |
| **Variable coverage (%)**   | Whether all relevant layers are mentioned        | Count mentions / expected total              | Text + keywords              |
| **Logical contradictions**  | Detect phrases inconsistent with CSV             | Rules (e.g., “clear sky” with `tcc>0.8`)     | CSV + text                   |
| **CLIPScore**               | Alignment between description and image          | Compute cosine similarity of CLIP embeddings | Text + corresponding image   |
| **Output length**           | Verbosity                                        | Count tokens/words in description            | Text                         |
| **n-gram repetition**       | Fluency (avoid redundancy)                       | % of repeated bigrams/trigrams               | Text                         |
| **Readability**             | Ease of reading                                  | Automated readability index                  | Text                         |
| **Grammar errors**          | Basic correctness                                | Automated checker (e.g., LanguageTool)       | Text                         |
| **Inference time**          | Generation speed                                 | Measure `end_time - start_time`              | Timer during inference       |
| **Memory/CPU/GPU usage**    | Computational resource footprint                 | Use `psutil` or GPU logs                     | System monitor               |
| **Monetary cost**           | Training/inference cost                          | Tokens × price (API) or electricity × hours  | Billing data or estimation   |
| **Energy cost (kWh)**       | Environmental impact                             | Avg. power consumption × time                | Hardware meter or cloud logs |
| **Latency per case**        | User-facing responsiveness                       | Time per description                         | Timer during inference       |
| **Model size (GB)**         | Storage footprint                                | Model checkpoint or deployment size          | Model info                   |


Inference time → necesitarías guardar timestamps (start_time, end_time) en el momento de generación.

Memory/CPU/GPU usage → requiere medir en tiempo real (psutil, nvidia-smi o logs del cluster).

Monetary cost → requiere o bien datos de facturación (API tokens × precio) o coste eléctrico × horas.

Energy cost (kWh) → requiere logs de consumo eléctrico medio × tiempo.

Latency per case → lo mismo que inference time, guardando tiempos por ejemplo con un cronómetro.

Model size (GB) → metadatos del modelo usado (checkpoint, despliegue).

In [1]:
import os
import re
import sys
import json
import math
import argparse
import numpy as np
import pandas as pd
from collections import Counter
from dataclasses import dataclass
from typing import Dict, Optional, List

In [7]:
KW = {
    "high_pos": [
        r"\bhigh cloud(s)?\b", r"\bhigh[- ]altitude cloud(s)?\b",
        r"\bcirrus\b", r"\bcirro(stratus|cumulus)\b",
        r"\b(hcc|high cloud cover)\b",
    ],
    "high_neg": [
        r"\bno (high[- ]altitude|high) cloud(s)?\b", r"\bwithout high cloud(s)?\b"
    ],
    "mid_pos": [
        r"\bmid(dle)?[- ]level cloud(s)?\b", r"\baltostratus\b", r"\baltocumulus\b",
        r"\b(mcc|mid(dle)? cloud cover)\b",
    ],
    "mid_neg": [
        r"\bno (mid(dle)?[- ]level|middle) cloud(s)?\b", r"\bwithout (mid|middle) cloud(s)?\b"
    ],
    "low_pos": [
        r"\blow[- ]level cloud(s)?\b", r"\bstratus\b", r"\bstratocumulus\b", r"\bfog\b",
        r"\b(lcc|low cloud cover)\b",
    ],
    "low_neg": [
        r"\bno (low[- ]level|low) cloud(s)?\b", r"\bwithout low cloud(s)?\b"
    ],
    # Clear / Overcast cues
    "clear": [
        r"\bclear sky\b", r"\bmostly clear\b", r"\bclear to fair\b", r"\bfair weather\b"
    ],
    "overcast": [
        r"\bovercast\b", r"\b(nearly|almost) (the )?entire sky (is )?covered\b", r"\bsky (is )?dominated\b"
    ]
}

In [2]:
def normalize_text(s: str) -> str:
    s = s or ""
    return re.sub(r"\s+", " ", s.strip().lower())

In [3]:
def word_tokens(text):
    return re.findall(r"[A-Za-zÀ-ÿ']+", (text or "").lower())

In [4]:
def sentence_split(text):
    return [s for s in re.split(r"[.!?]+", (text or "")) if s.strip()]

In [5]:
def readability_proxy(text):
    """
    Language-agnostic readability proxy (0-100; higher is easier).
    """
    sents = sentence_split(text)
    words = word_tokens(text)
    if not sents or not words:
        return np.nan
    avg_sent_len = len(words) / len(sents)
    avg_word_len = sum(len(w) for w in words) / len(words)
    score = 100 - (avg_sent_len - 15) * 2 - (avg_word_len - 5) * 10
    return max(0, min(100, score))

In [6]:
def ngram_repetition_ratio(text, n=2):
    tokens = word_tokens(text)
    if len(tokens) < n+1:
        return 0.0
    ngrams = [" ".join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    counts = Counter(ngrams)
    repeats = sum(c-1 for c in counts.values() if c > 1)
    return repeats / max(1, len(ngrams))

In [ ]:
def any_match(patterns, text):
    return any(re.search(p, text) for p in patterns)

In [ ]:
def presence_from_text(text, layer):
    text = normalize_text(text)
    if layer == "high":
        if any_match(KW["high_neg"], text): return False
        if any_match(KW["high_pos"], text): return True
    elif layer == "mid":
        if any_match(KW["mid_neg"], text): return False
        if any_match(KW["mid_pos"], text): return True
    elif layer == "low":
        if any_match(KW["low_neg"], text): return False
        if any_match(KW["low_pos"], text): return True
    return None


In [ ]:
def extract_numeric_from_text_for_layer(text, layer_name):
    """
    Extract an explicit numeric for a layer if present, return proportion [0,1], else None.
    """
    text = normalize_text(text)
    if layer_name == "hcc":
        layer_patterns = [r"high cloud (cover|coverage)\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*%",
                          r"\bhcc\b\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*(%|)",
                          r"high cloud(s)?.*?(\d+(?:\.\d+)?)\s*%",
                          r"\bnubes altas.*?(\d+(?:\.\d+)?)\s*%"]
    elif layer_name == "mcc":
        layer_patterns = [r"(mid|middle|midlevel|mid-level) cloud (cover|coverage)\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*%",
                          r"\bmcc\b\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*(%|)",
                          r"(mid|middle).*?(\d+(?:\.\d+)?)\s*%",
                          r"\bnubes medias.*?(\d+(?:\.\d+)?)\s*%"]
    elif layer_name == "lcc":
        layer_patterns = [r"(low|lowlevel|low-level) cloud (cover|coverage)\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*%",
                          r"\blcc\b\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*(%|)",
                          r"(low).*?(\d+(?:\.\d+)?)\s*%",
                          r"\bnubes bajas.*?(\d+(?:\.\d+)?)\s*%"]
    else:
        return None
    for pat in layer_patterns:
        m = re.search(pat, text)
        if m:
            nums = [g for g in m.groups() if g and re.match(r"^\d+(\.\d+)?$", g)]
            if nums:
                val = float(nums[-1])
                return val if 0 <= val <= 1 else val / 100.0
    return None

In [ ]:
def compute_f1_from_confusion(tp, fp, fn):
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2*prec*rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return prec, rec, f1

In [ ]:
def logical_contradictions_row(row):
    desc = normalize_text(row.get("description", ""))
    tcc = row.get("tcc", np.nan)
    contras = 0
    if not pd.isna(tcc):
        if tcc > 0.8 and any_match(KW["clear"], desc):
            contras += 1
        if tcc < 0.2 and any_match(KW["overcast"], desc):
            contras += 1
    flags = {
        "high": row.get("hcc", np.nan) > 0.1 if not pd.isna(row.get("hcc", np.nan)) else None,
        "mid":  row.get("mcc", np.nan) > 0.1 if not pd.isna(row.get("mcc", np.nan)) else None,
        "low":  row.get("lcc", np.nan) > 0.1 if not pd.isna(row.get("lcc", np.nan)) else None,
    }
    preds = {
        "high": presence_from_text(desc, "high"),
        "mid":  presence_from_text(desc, "mid"),
        "low":  presence_from_text(desc, "low"),
    }
    for k in ["high","mid","low"]:
        if flags[k] is None or preds[k] is None:
            continue
        if flags[k] and preds[k] is False:
            contras += 1
        if (not flags[k]) and preds[k] is True:
            contras += 1
    return contras

In [ ]:
# ------------------------ Optional: CLIPScore ------------------------

def compute_clip_scores(df, image_col, text_col, model_name="openai/clip-vit-base-patch32", device=None, batch_size=8,
                        norm_low=0.10, norm_high=0.40, normalize=False):
    """
    Compute cosine similarity between text and image using CLIP.
    If normalize=True, also map to [0,100] using [norm_low, norm_high] bounds.
    """
    try:
        import torch
        from transformers import CLIPProcessor, CLIPModel
        from PIL import Image
    except Exception as e:
        print("[WARN] CLIP dependencies not available. Skipping CLIPScore. Error:", e)
        return [np.nan]*len(df), [np.nan]*len(df) if normalize else None

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    try:
        model = CLIPModel.from_pretrained(model_name).to(device).eval()
        processor = CLIPProcessor.from_pretrained(model_name)
    except Exception as e:
        print("[WARN] Could not load CLIP model. Skipping:", e)
        return [np.nan]*len(df), [np.nan]*len(df) if normalize else None

    scores = []
    norm_scores = []
    with torch.inference_mode():
        i = 0
        N = len(df)
        while i < N:
            batch = df.iloc[i:i+batch_size]
            texts = batch[text_col].fillna("").astype(str).tolist()
            images = []
            for p in batch[image_col].fillna("").astype(str).tolist():
                try:
                    from PIL import Image
                    img = Image.open(p).convert("RGB")
                except Exception:
                    # If image is missing/unreadable, use a black placeholder (won't match text well)
                    img = Image.new("RGB", (224,224), color=(0,0,0))
                images.append(img)
            inputs = processor(text=texts, images=images, return_tensors="pt", padding=True).to(device)
            outputs = model(**inputs)
            img_emb = outputs.image_embeds / outputs.image_embeds.norm(dim=-1, keepdim=True)
            txt_emb = outputs.text_embeds / outputs.text_embeds.norm(dim=-1, keepdim=True)
            cos = (txt_emb * img_emb).sum(dim=-1)  # [batch]
            batch_scores = cos.detach().cpu().tolist()
            scores.extend(batch_scores)
            if normalize:
                # Map to [0,100] using linear scaling with clipping
                s = np.array(batch_scores, dtype=float)
                s = np.clip(s, norm_low, norm_high)
                s = (s - norm_low) / (norm_high - norm_low) * 100.0
                norm_scores.extend(s.tolist())
            i += batch_size
    return scores, (norm_scores if normalize else None)

In [ ]:
# ------------------------ Optional: Grammar errors ------------------------

def setup_language_tool(lang_code: str):
    """
    Try to set up a LanguageTool checker for a given language code (e.g., 'en-US', 'es-ES').
    Returns a callable that counts errors, or None if unavailable.
    """
    try:
        import language_tool_python as ltp
    except Exception as e:
        print("[WARN] language_tool_python not available:", e)
        return None
    try:
        tool = ltp.LanguageTool(lang_code)
    except Exception as e:
        print(f"[WARN] Could not init LanguageTool for {lang_code}:", e)
        return None
    def count_errors(text: str) -> int:
        try:
            return len(tool.check(text or ""))
        except Exception:
            return np.nan
    return count_errors

In [ ]:
# ------------------------ Performance metrics (from logs) ------------------------

def compute_performance_metrics(logs_df: pd.DataFrame, prices: Optional[Dict[str, Dict[str, float]]] = None):
    """
    Expect logs_df columns (some optional):
      model, case_id, start_time, end_time, tokens_in, tokens_out, power_w_avg, energy_kwh, memory_mb_max, gpu_mem_mb_max
    Times in ISO or epoch; tokens as integers.
    Prices: dict like {"openai:gpt-4o":{"input":5.0,"output":15.0}} in $ per 1k tokens.
    """
    perf = logs_df.copy()
    for col in ["start_time","end_time"]:
        if col in perf.columns:
            perf[col] = pd.to_datetime(perf[col], errors="coerce")
    if set(["start_time","end_time"]).issubset(perf.columns):
        perf["latency_s"] = (perf["end_time"] - perf["start_time"]).dt.total_seconds()
    if prices and set(["model","tokens_in","tokens_out"]).issubset(perf.columns):
        def cost_row(r):
            p = prices.get(str(r["model"]), None)
            if not p: return np.nan
            cin = (r.get("tokens_in", 0) or 0) / 1000.0 * p.get("input", 0.0)
            cout = (r.get("tokens_out", 0) or 0) / 1000.0 * p.get("output", 0.0)
            return cin + cout
        perf["monetary_cost_usd"] = perf.apply(cost_row, axis=1)
    if "power_w_avg" in perf.columns and "latency_s" in perf.columns:
        perf["energy_kwh"] = perf["power_w_avg"] * perf["latency_s"] / 3600.0 / 1000.0
    return perf

In [ ]:
# ------------------------ Main pipeline ------------------------

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--input_csv", required=True, help="Path to input CSV with columns including description,hcc,mcc,lcc,tcc,path_image")
    ap.add_argument("--out_dir", required=True, help="Directory to write outputs")
    ap.add_argument("--compute_clip", action="store_true", help="Compute CLIPScore (requires transformers+torch and valid images)")
    ap.add_argument("--clip_model", default="openai/clip-vit-base-patch32", help="CLIP model name")
    ap.add_argument("--clip_norm_low", type=float, default=0.10, help="Lower bound for CLIP normalization")
    ap.add_argument("--clip_norm_high", type=float, default=0.40, help="Upper bound for CLIP normalization")
    ap.add_argument("--clip_batch_size", type=int, default=8, help="CLIP batch size")
    ap.add_argument("--grammar_lang", default=None, help="Language code for grammar check (e.g., 'en-US' or 'es-ES')")
    ap.add_argument("--logs_csv", default=None, help="Optional: CSV with runtime logs to compute performance metrics")
    ap.add_argument("--prices_json", default=None, help="Optional: JSON mapping model->input/output $ per 1k tokens")
    args = ap.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    df = pd.read_csv(args.input_csv)

    for col in ["description","hcc","mcc","lcc","tcc","path_image"]:
        if col not in df.columns:
            df[col] = np.nan

    df["_desc_norm"] = df["description"].fillna("").astype(str)

    # Variable coverage
    for layer in ["high","mid","low"]:
        df[f"mention_{layer}"] = df["_desc_norm"].apply(lambda t: presence_from_text(t, layer))

    def coverage_percent(row):
        vals = [row[f"mention_{l}"] for l in ["high","mid","low"]]
        mentioned = sum(v is not None for v in vals)
        return 100.0 * mentioned / 3.0

    df["variable_coverage_pct"] = df.apply(coverage_percent, axis=1)

    # Cloud classification F1
    def confusion_for_layer(df_layer, layer, thresh=0.1):
        col = {"high":"hcc","mid":"mcc","low":"lcc"}[layer]
        truth = df_layer[col]
        present_truth = truth.apply(lambda x: (not pd.isna(x)) and (x > thresh))
        pred = df_layer[f"mention_{layer}"]
        mask = pred.notna()
        tp = int(((pred[mask] == True) & (present_truth[mask] == True)).sum())
        fp = int(((pred[mask] == True) & (present_truth[mask] == False)).sum())
        fn = int(((pred[mask] == False) & (present_truth[mask] == True)).sum())
        return tp, fp, fn, int(mask.sum())

    layer_stats = {}
    for layer in ["high","mid","low"]:
        tp, fp, fn, n = confusion_for_layer(df, layer)
        prec, rec, f1 = compute_f1_from_confusion(tp, fp, fn)
        layer_stats[layer] = {"tp":tp,"fp":fp,"fn":fn,"n_eval":n,"precision":prec,"recall":rec,"f1":f1}
    macro_f1 = float(np.mean([layer_stats[l]["f1"] for l in ["high","mid","low"]]))

    # Logical contradictions
    df["logical_contradictions_count"] = df.apply(logical_contradictions_row, axis=1)

    # Output length & repetition
    df["word_count"] = df["_desc_norm"].apply(lambda t: len(word_tokens(t)))
    df["bigram_repeat_ratio"] = df["_desc_norm"].apply(lambda t: ngram_repetition_ratio(t, n=2))
    df["trigram_repeat_ratio"] = df["_desc_norm"].apply(lambda t: ngram_repetition_ratio(t, n=3))

    # Readability
    df["readability_score_0_100"] = df["_desc_norm"].apply(readability_proxy)

    # Numeric coverage MAE (only if numbers are present)
    for layer, col in [("hcc","hcc"),("mcc","mcc"),("lcc","lcc")]:
        df[f"{col}_pred_from_text"] = df["_desc_norm"].apply(lambda t: extract_numeric_from_text_for_layer(t, layer))

    def mae_col(df_in, col_true, col_pred):
        mask = df_in[col_pred].notna() & df_in[col_true].notna()
        if mask.sum() == 0:
            return np.nan, 0
        return float(np.mean(np.abs(df_in.loc[mask, col_true] - df_in.loc[mask, col_pred]))), int(mask.sum())

    mae_hcc, n_hcc = mae_col(df, "hcc", "hcc_pred_from_text")
    mae_mcc, n_mcc = mae_col(df, "mcc", "mcc_pred_from_text")
    mae_lcc, n_lcc = mae_col(df, "lcc", "lcc_pred_from_text")

    # Optional CLIPScore
    df["clipscore"] = np.nan
    df["clipscore_norm_0_100"] = np.nan
    if args.compute_clip and "path_image" in df.columns and df["path_image"].notna().any():
        scores, norm_scores = compute_clip_scores(
            df, "path_image", "description",
            model_name=args.clip_model,
            batch_size=args.clip_batch_size,
            norm_low=args.clip_norm_low,
            norm_high=args.clip_norm_high,
            normalize=True
        )
        df["clipscore"] = scores
        df["clipscore_norm_0_100"] = norm_scores

    # Optional grammar errors
    if args.grammar_lang:
        count_fn = setup_language_tool(args.grammar_lang)
        if count_fn is not None:
            df["grammar_errors"] = df["_desc_norm"].apply(lambda t: count_fn(t))
        else:
            df["grammar_errors"] = np.nan
    else:
        df["grammar_errors"] = np.nan

    # Aggregate summary
    summary = {
        "macro_f1_layers": macro_f1,
        "f1_high": layer_stats["high"]["f1"],
        "f1_mid": layer_stats["mid"]["f1"],
        "f1_low": layer_stats["low"]["f1"],
        "precision_high": layer_stats["high"]["precision"],
        "recall_high": layer_stats["high"]["recall"],
        "precision_mid": layer_stats["mid"]["precision"],
        "recall_mid": layer_stats["mid"]["recall"],
        "precision_low": layer_stats["low"]["precision"],
        "recall_low": layer_stats["low"]["recall"],
        "avg_variable_coverage_pct": float(df["variable_coverage_pct"].mean()),
        "avg_logical_contradictions": float(df["logical_contradictions_count"].mean()),
        "avg_word_count": float(df["word_count"].mean()),
        "avg_bigram_repeat_ratio": float(df["bigram_repeat_ratio"].mean()),
        "avg_trigram_repeat_ratio": float(df["trigram_repeat_ratio"].mean()),
        "avg_readability_score_0_100": float(df["readability_score_0_100"].mean()),
        "mae_hcc": mae_hcc, "n_hcc_used": n_hcc,
        "mae_mcc": mae_mcc, "n_mcc_used": n_mcc,
        "mae_lcc": mae_lcc, "n_lcc_used": n_lcc,
        "avg_clipscore": float(pd.to_numeric(df["clipscore"], errors="coerce").mean()),
        "avg_clipscore_norm_0_100": float(pd.to_numeric(df["clipscore_norm_0_100"], errors="coerce").mean()),
        "avg_grammar_errors": float(pd.to_numeric(df["grammar_errors"], errors="coerce").mean()),
    }

    # Optional performance metrics from logs
    perf_df = None
    if args.logs_csv and os.path.exists(args.logs_csv):
        try:
            logs_df = pd.read_csv(args.logs_csv)
            prices = json.loads(args.prices_json) if args.prices_json else None
            perf_df = compute_performance_metrics(logs_df, prices=prices)
            perf_out = os.path.join(args.out_dir, "performance_metrics.csv")
            perf_df.to_csv(perf_out, index=False)
            summary["have_performance_logs"] = True
        except Exception as e:
            print("[WARN] Could not compute performance metrics:", e)
            summary["have_performance_logs"] = False
    else:
        summary["have_performance_logs"] = False

    # Save outputs
    perrow_out = os.path.join(args.out_dir, "per_row_metrics.csv")
    df.to_csv(perrow_out, index=False)

    summary_out = os.path.join(args.out_dir, "summary_metrics.json")
    with open(summary_out, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    # Small console print
    print("== Summary ==")
    for k, v in summary.items():
        print(f"{k}: {v}")
    print("\nWrote:")
    print(" -", perrow_out)
    if perf_df is not None:
        print(" -", perf_out)
    print(" -", summary_out)

In [ ]:
if __name__ == "__main__":
    main()
